In [ ]:
# SDPA implementation
import torch
import torch.nn.functional as F
import math
# Efficient implementation equivalent to the following:
# query: [B,H,L,D]
# key: [B,H,S,D]
# value: [B,H,S,Dv]

def scaled_dot_product_attention(query, key, value, attn_mask=None, dropout_p=0.0,
        is_causal=False, scale=None, enable_gqa=False) -> torch.Tensor:
        L, S = query.size(-2), key.size(-2)
        scaler = 1/math.sqrt(query.size(-1)) if scale is None else scale
        attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
        if is_causal:
                assert attn_mask is None
                causal_mask = torch.ones((L,S), dtype = torch.bool,device=query.device).tril()
                attn_bias = attn_bias.masked_fill(~causal_mask, float("-inf"))
        if attn_mask is not None:
                if attn_mask.dtype == torch.bool:
                        attn_bias = torch.zeros_like(attn_mask, dtype=torch.float)
                        attn_bias.masked_fill_(~attn_mask, float("-inf"))
                else:
                        attn_bias = attn_mask + attn_bias
        if enable_gqa:
                key = key.repeat_interleave(query.size(-3)//key.size(-3), dim=-3)
                value = value.repeat_interleave(query.size(-3)//value.size(-3), dim=-3)

        attn_weight = query @ key.transpose(-2,-1) * scaler
        attn_weight = attn_weight + attn_bias
        attn_weight = torch.softmax(attn_weight,dim=-1)
        attn_weight = torch.dropout(attn_weight, dropout_p,train=True)
        attn_value = attn_weight @ value
        return attn_value

In [ ]:
def multi_head_attention_forward(
    x: torch.Tensor,           # [B, L, E]
    num_heads: int,
    w_q: torch.Tensor,         # [E, E]
    w_k: torch.Tensor,         # [E, E]
    w_v: torch.Tensor,         # [E, E]
    w_o: torch.Tensor,         # [E, E]
    attn_mask: torch.Tensor | None = None,
) -> torch.Tensor:
    B, L, E = x.shape
    q,k,v = x @ w_q, x @ w_k, x @ w_v
    H = num_heads
    assert E % H ==0
    D = E // H
    q = q.view(B,L,H,D).transpose(-3,-2)
    k = k.view(B,L,H,D).transpose(-3,-2)
    v = v.view(B,L,H,D).transpose(-3,-2)
    attn_value = scaled_dot_product_attention(q,k,v,is_causal=True, attn_mask=attn_mask)
    attn_value = attn_value.transpose(-3,-2).contiguous().view(B,L,E)
    return attn_value @ w_o



In [ ]:
def rotate_half(x: torch.Tensor) -> torch.Tensor: #[B, H, L, D]
    # [a,b,c,d] -> [-c, -d, a, b]
    x1 = x[...,:x.size(-1)//2]
    x2 = x[...,x.size(-1)//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(
    q: torch.Tensor,
    k: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
    unsqueeze_dim: int = 1,
) -> tuple[torch.Tensor, torch.Tensor]:
    # q: [B, H, L, D]
    # k: [B, H, L, D]
    # cos: [B, L, D]
    # sin: [B, L, D]
    cos = cos.unsqueeze(unsqueeze_dim) # [B, 1, L, D]
    sin = sin.unsqueeze(unsqueeze_dim) # [B, 1, L, D]
    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin
    return q , k